# 模型生成结果（中）

### 50repo数据集——历史合并冲突

In [ ]:
#qwen获得top5解决方案
import json
import random
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        #random.shuffle(json_data) # 随机洗牌顺序
        processed_count = 0  # 用于记录已处理的数据数量

        for i, item in enumerate(json_data):
            # if i + 1 < 1022: # 从异常发生点接着运行
            #     continue
            # if i + 1 == 501: # 从异常发生点接着运行
            #     break

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = repo_url.rstrip('/').split('/')[-1].split('-')[0]
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            global_id, related_c, related_r = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            # print(f"\n<--------------------------------------------示例{i + 1}(line_no_T5)：-------------------------------------------->")
            # item['line_no_T5'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer_T5)：-------------------------------------------->")
            item['line_withRefer_T5'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

            # 每处理完一条数据，计数器加1
            processed_count += 1
            
            # 每处理10条数据，自动保存一次
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as output_file:
                    json.dump(json_data, output_file, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()

# request_gpt(input = "YOUR_PROJECT_DIR/dataset/congra/json/aosp.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce2/congra/json/aosp.json") 
    
request_gpt(input = "YOUR_PROJECT_DIR/produce2/50repo/json/fred.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce2/50repo/json/fred.json") 

In [ ]:
# deepseek获得top5解决方案
import json
import random
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from deepseek_api import deepSeek
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        #random.shuffle(json_data) # 随机洗牌顺序
        processed_count = 0  # 用于记录已处理的数据数量

        for i, item in enumerate(json_data):
            if i + 1 < 769: # 从异常发生点接着运行
                continue

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = repo_url.rstrip('/').split('/')[-1].split('-')[0]
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            global_id, related_c, related_r = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            # print(f"\n<--------------------------------------------示例{i + 1}(line_no_T5)：-------------------------------------------->")
            # item['line_no_T5'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer_T5)：-------------------------------------------->")
            item['line_withRefer_T5'] = deepSeek(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

            # 每处理完一条数据，计数器加1
            processed_count += 1
            
            # 每处理10条数据，自动保存一次
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as output_file:
                    json.dump(json_data, output_file, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()

# request_gpt(input = "YOUR_PROJECT_DIR/dataset/50repo/json/jgralab.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/deepseek_produce/50repo/json/jgralab.json") 
    
request_gpt(input = "YOUR_PROJECT_DIR/deepseek_produce/50repo/json/jgralab.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/deepseek_produce/50repo/json/jgralab.json") 

## mergebert整后80%的数据

In [ ]:
#qwen获得top5解决方案
import json
import random
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        #random.shuffle(json_data) # 随机洗牌顺序
        processed_count = 0  # 用于记录已处理的数据数量

        for i, item in enumerate(json_data):
            # if i + 1 < 244: # 从异常发生点接着运行
            #     continue
            # if i + 1 == 350: # 从异常发生点接着运行
            #     break

            id = item.get('id', '')
            # repo_url = item.get('repo', '')
            # repo_name = repo_url.rstrip('/').split('/')[-1].split('-')[0]
            # repo_name = (repo_url.split('/')[-1].split('-')[0] + "_time") if repo_url else ''
            repo_name = "mergebert_80"
            print(repo_name)
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            global_id, related_c, related_r = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            # print(f"\n<--------------------------------------------示例{i + 1}(line_no_T5)：-------------------------------------------->")
            # item['line_no_T5'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer_T5)：-------------------------------------------->")
            item['line_withRefer_T5'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

            # 每处理完一条数据，计数器加1
            processed_count += 1
            
            # 每处理10条数据，自动保存一次
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as output_file:
                    json.dump(json_data, output_file, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()


In [ ]:
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autopsy_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/autopsy_time20.json") 

In [ ]:

request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/autorest_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/autorest_time20.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/azure_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/azure_time20.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/cloudstack_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/cloudstack_time20.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/Galacticraft_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/Galacticraft_time20.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/molgenis_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/molgenis_time20.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/orientdb_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/orientdb_time20.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/rumble_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/rumble_time20.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/sis_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/sis_time20.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/SOS_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/SOS_time20.json")
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert_20/spring_time20.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce/mergebert_20/spring_time20.json")

### MergeBERT数据集——历史合并冲突

In [ ]:
#qwen获得top5解决方案
import json
import random
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        #random.shuffle(json_data) # 随机洗牌顺序
        processed_count = 0  # 用于记录已处理的数据数量

        for i, item in enumerate(json_data):
            if i + 1 < 244: # 从异常发生点接着运行
                continue
            if i + 1 == 350: # 从异常发生点接着运行
                break

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            # repo_name = repo_url.rstrip('/').split('/')[-1].split('-')[0]
            repo_name = (repo_url.split('/')[-1].split('-')[0] + "_time") if repo_url else ''
            print(repo_name)
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            global_id, related_c, related_r = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            print(f"\n<--------------------------------------------示例{i + 1}(line_no_T5)：-------------------------------------------->")
            item['line_no_T5'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer_T5)：-------------------------------------------->")
            item['line_withRefer_T5'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

            # 每处理完一条数据，计数器加1
            processed_count += 1
            
            # 每处理10条数据，自动保存一次
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as output_file:
                    json.dump(json_data, output_file, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()

# request_gpt(input = "YOUR_PROJECT_DIR/dataset/congra/json/aosp.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce2/congra/json/aosp.json") 
    
# request_gpt(input = "YOUR_PROJECT_DIR/produce2/50repo/json/fred.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce2/50repo/json/fred.json") 
request_gpt(input = "YOUR_PROJECT_DIR/produce2/mergebert/json/autopsy_time.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce2/mergebert/json/autopsy_time.json") 

## 对比

In [ ]:
#qwen获得top5解决方案
import json
import random
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        #random.shuffle(json_data) # 随机洗牌顺序
        processed_count = 0  # 用于记录已处理的数据数量

        for i, item in enumerate(json_data):
            # if i + 1 < 1022: # 从异常发生点接着运行
            #     continue
            if i + 1 == 350: # 从异常发生点接着运行
                break

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = repo_url.rstrip('/').split('/')[-1].split('-')[0]
            print(repo_name)
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            global_id, related_c, related_r = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            # print(f"\n<--------------------------------------------示例{i + 1}(line_no_T5)：-------------------------------------------->")
            # item['line_no_T5'] = qwen(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer_T5)：-------------------------------------------->")
            item['line_withRefer_T5'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

            # 每处理完一条数据，计数器加1
            processed_count += 1
            
            # 每处理10条数据，自动保存一次
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as output_file:
                    json.dump(json_data, output_file, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()

# request_gpt(input = "YOUR_PROJECT_DIR/dataset/congra/json/aosp.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce2/congra/json/aosp.json") 
    
# request_gpt(input = "YOUR_PROJECT_DIR/produce2/50repo/json/fred.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce2/50repo/json/fred.json") 
request_gpt(input = "YOUR_PROJECT_DIR/dataset/mergebert/json/autopsy_time.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce2/mergebert/json/autopsyv3.json") 

## congra数据集——历史合并冲突

### qwen

In [ ]:
import json
import random
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        #random.shuffle(json_data) # 随机洗牌顺序
        processed_count = 0  # 用于记录已处理的数据数量

        for i, item in enumerate(json_data):
            if i + 1 < 416: # 从异常发生点接着运行
                continue
            if i + 1 == 501: # 从异常发生点接着运行
                break

            id = item.get('id', '')
            repo_url = item.get('repo', '')

            # 正确提取仓库名：aosp\\conflict_files_0 → aosp
            repo_name = repo_url.split('\\')[0] if '\\' in repo_url else repo_url.split('/')[-1].split('-')[0]

            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            global_id, related_c, related_r = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer_T5)：-------------------------------------------->")
            item['line_withRefer_T5'] = qwen(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

            processed_count += 1
            
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as output_file:
                    json.dump(json_data, output_file, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}Checkmark: 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error: An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warning: ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()

# request_gpt(input = "YOUR_PROJECT_DIR/dataset/congra/json/aosp.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/produce2/congra/json/aosp.json") 
    
request_gpt(input = "YOUR_PROJECT_DIR/produce2/congra/json/aosp.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/produce2/congra/json/aosp.json") 

### deepseek

In [ ]:
import json
import random
from utils import simplePrompt, prompt1
from colorama import Fore, Style
from deepseek_api import deepSeek
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"

def request_gpt(input, output):
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        random.shuffle(json_data) # 随机洗牌顺序
        processed_count = 0  # 用于记录已处理的数据数量

        for i, item in enumerate(json_data):
            # if i + 1 < 769: # 从异常发生点接着运行
            #     continue

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = repo_url.rstrip('/').split('/')[-1].split('-')[0]
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            global_id, related_c, related_r = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            print(f"\n<--------------------------------------------示例{i + 1}(line_no_T5)：-------------------------------------------->")
            item['line_no_T5'] = deepSeek(question=simplePrompt(diff_conflict), number=3, truth=res_region)

            print(f"\n<--------------------------------------------示例{i + 1}(line_withRefer_T5)：-------------------------------------------->")
            item['line_withRefer_T5'] = deepSeek(question=prompt1(refer_c1, refer_r1, diff_conflict), number=5, truth=res_region)

            # 每处理完一条数据，计数器加1
            processed_count += 1
            
            # 每处理10条数据，自动保存一次
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as output_file:
                    json.dump(json_data, output_file, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()

request_gpt(input = "YOUR_PROJECT_DIR/dataset/congra/json/aosp.json", # 处理异常的时候这里也要改成相同的文件名
            output= "YOUR_PROJECT_DIR/deepseek_produce/congra/json/aosp.json") 
    
# request_gpt(input = "YOUR_PROJECT_DIR/deepseek_produce/50repo/json/jgralab.json", # 处理异常的时候这里也要改成相同的文件名
#             output= "YOUR_PROJECT_DIR/deepseek_produce/50repo/json/jgralab.json") 

# 模型生成结果（后）

## 多模型排序的前置操作+后续操作

In [ ]:
import json
import os

# 输入文件路径
qwen_file = r"YOUR_PROJECT_DIR\dataset_multillm\fred\fred_qwen.json"
deepseek_file = r"YOUR_PROJECT_DIR\dataset_multillm\fred\fred_deepseek.json"

# 输出文件路径
output_file = os.path.join(os.path.dirname(qwen_file), "fred_merged.json")

# 加载文件
with open(qwen_file, "r", encoding="utf-8") as f:
    qwen_data = json.load(f)

with open(deepseek_file, "r", encoding="utf-8") as f:
    deepseek_data = json.load(f)

# 用 deepseek 的 id 建立索引
deepseek_index = {item["id"]: item for item in deepseek_data}

merged = []

for q_item in qwen_data:
    item_id = q_item["id"]

    # 仅处理 deepseek 中存在相同 id 的记录
    if item_id in deepseek_index:
        d_item = deepseek_index[item_id]

        # 复制 qwen 原有字段
        new_item = dict(q_item)

        # 合并两个模型的 line_withRefer_T5，形成一个平铺列表
        list_qwen = q_item.get("line_withRefer_T5", [])
        list_deepseek = d_item.get("line_withRefer_T5", [])
        merged_list = list_qwen + list_deepseek

        new_item["line_withRefer_T5_merged"] = merged_list

        merged.append(new_item)

# 写文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

print("合并完成，生成文件：", output_file)


In [ ]:
import json
from difflib import SequenceMatcher

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    """判断 target 是否含有 source 中独有 token"""
    return any(tok.strip() and tok in target for tok in source.split())

def score_candidate(candidate, a, b, o):
    score = 0

    # 基础语义相似度
    score += 0.5 * sim(candidate, a)
    score += 0.5 * sim(candidate, b)

    # base版本越像，说明没融合，做惩罚
    base_similarity = sim(candidate, o)
    score -= 0.7 * base_similarity

    # 额外结构特征规则
    if contains_tokens(a, candidate) and contains_tokens(b, candidate):
        score += 2  # 代表跨分支融合
    elif contains_tokens(a, candidate) or contains_tokens(b, candidate):
        score += 0.5  # 至少来自其中一个版本

    # 过短代码惩罚（避免损坏代码）
    score -= abs(len(a) - len(candidate)) * 0.0005

    return score
input_file  = r"YOUR_PROJECT_DIR\dataset_multillm\fred\fred_merged.json"          # ← 输入文件
output_file = r"YOUR_PROJECT_DIR\dataset_multillm\fred\fred_merged_sorted.json" 

with open(input_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

for item in dataset:
    if "line_withRefer_T5_merged" not in item:
        continue

    a = item.get("a_contents", "")
    b = item.get("b_contents", "")
    o = item.get("o_contents", "")

    candidates = item["line_withRefer_T5_merged"]
    
    ranked = sorted(
        candidates, 
        key=lambda c: score_candidate(c, a, b, o), 
        reverse=True
    )

    item["line_withRefer_T5_merged"] = ranked

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print("排序完成 →", output_file)


## 效果较好的基于语义的重排序

In [ ]:
import json
from pathlib import Path
from difflib import SequenceMatcher


def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()


def contains_tokens(source, target):
    """判断 target 是否含有 source 中独有 token"""
    return any(tok.strip() and tok in target for tok in source.split())


def score_candidate(candidate, a, b, o):
    score = 0

    # 基础语义相似度
    score += 0.5 * sim(candidate, a)
    score += 0.5 * sim(candidate, b)

    # base版本越像，说明没融合，做惩罚
    base_similarity = sim(candidate, o)
    score -= 0.7 * base_similarity

    # 额外结构特征规则
    if contains_tokens(a, candidate) and contains_tokens(b, candidate):
        score += 2  # 代表跨分支融合
    elif contains_tokens(a, candidate) or contains_tokens(b, candidate):
        score += 0.5  # 至少来自其中一个版本

    # 过短代码惩罚（避免损坏代码）
    score -= abs(len(a) - len(candidate)) * 0.0005

    return score


def rerank_json_file(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    for item in dataset:
        if "line_withRefer1" not in item:
            continue

        a = item.get("a_contents", "")
        b = item.get("b_contents", "")
        o = item.get("o_contents", "")
        candidates = item["line_withRefer1"]

        item["line_withRefer1"] = sorted(
            candidates,
            key=lambda c: score_candidate(c, a, b, o),
            reverse=True,
        )

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

    print("排序完成 →", output_file)


source_dir = Path(r"dataset_split/outputs/qwen")
output_dir = source_dir / "rerank"

for input_path in source_dir.glob("*.json"):
    if input_path.parent == output_dir:
        continue

    output_path = output_dir / f"{input_path.stem}_sorted.json"
    rerank_json_file(input_path, output_path)


## 大模型对解决方案进行排序

### qwen

In [ ]:
# 调用 Qwen 对top5 进行排序
import json
import re
from colorama import Fore, Style
from alibaba_api import qwen  
from utils import prompt_sort5,extract_sorted_candidates

def request_gpt(input, output):

    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
    except Exception as e:
        print(f"{Fore.RED}无法打开输入文件: {e}{Style.RESET_ALL}")
        return

    processed_count = 0

    for i, item in enumerate(json_data):
        print(f"\n<--------------------------------------------示例 {i + 1} (line_withRefer_T5)：-------------------------------------------->")

        try:
            # 三个候选
            line_refs = item.get("line_withRefer_T5", [])
            if len(line_refs) < 5:
                print(f"{Fore.YELLOW}Warning: line_withRefer_T5 不足 3 个候选。继续处理（会用空占位）。{Style.RESET_ALL}")
                line_refs = line_refs + [''] * (5 - len(line_refs))

            candidate1, candidate2, candidate3, candidate4, candidate5 = line_refs[:5]

            # 构造 diff_conflict（你指定的版本）
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + \
                       '\nbase_content\n' + item.get('o_contents', '') + \
                       '\nincoming_change\n' + item.get('b_contents', '')

            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<') \
                                    .replace('base_content', '|||||||') \
                                    .replace('incoming_change', '=======') + '\n>>>>>>>'

            # 模型真实 resolution（你传给 Qwen 做提示用）
            res_region = item.get("res_region", "")

            # 构造 prompt
            prompt = prompt_sort5(candidate1, candidate2, candidate3,candidate4,candidate5, diff_conflict)

            # 调模型
            model_output_list = qwen(question=prompt, number=1, truth=res_region)

            sorted_candidates = None

            if not model_output_list:
                sorted_candidates = []
            else:
                first = model_output_list[0]

                if isinstance(first, str):
                    raw_text = first.strip()
                    # 若以 '[' 开头，优先 JSON 解析
                    if raw_text.startswith('['):
                        try:
                            parsed = json.loads(raw_text)
                            if isinstance(parsed, list):
                                extracted = []
                                for elem in parsed:
                                    if isinstance(elem, dict) and len(elem) == 1:
                                        extracted.append(list(elem.values())[0])
                                    else:
                                        extracted.append(str(elem))
                                sorted_candidates = extracted
                        except:
                            sorted_candidates = extract_sorted_candidates(raw_text)
                    else:
                        sorted_candidates = extract_sorted_candidates(raw_text)

                elif isinstance(first, list):
                    # 极少情况：第一层就是候选数组
                    extracted = []
                    for elem in first:
                        if isinstance(elem, dict):
                            vals = list(elem.values())
                            if vals:
                                extracted.append(vals[0])
                        else:
                            extracted.append(str(elem))
                    sorted_candidates = extracted

                else:
                    raw_text = str(first)
                    sorted_candidates = extract_sorted_candidates(raw_text)

            if sorted_candidates is None:
                sorted_candidates = []

            # 写回
            item["sorted_T5"] = sorted_candidates

            processed_count += 1

            # 每 10 条保存
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as out:
                    json.dump(json_data, out, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

        except Exception as e:
            print(f"{Fore.RED}处理示例 {i + 1} 时发生异常: {e}{Style.RESET_ALL}")
            item.setdefault("_error", str(e))
            item["sorted_T5"] = []
            continue

    # 最终保存
    with open(output, 'w', encoding='utf-8') as out:
        json.dump(json_data, out, indent=2, ensure_ascii=False)

    print(f"{Fore.GREEN}完成：共处理 {processed_count} 条样本，结果已保存至 {output}{Style.RESET_ALL}")


#############################################
# 5. 执行入口
#############################################

if __name__ == "__main__":
    request_gpt(
        input="YOUR_PROJECT_DIR/produce2/congra/json/aosp.json",
        output="YOUR_PROJECT_DIR/produce2/congra/json/aosp_qwen_sort.json"
    )


In [ ]:
# 调用 Qwen 对top3 进行排序
import json
import re
from colorama import Fore, Style
from alibaba_api import qwen  
from utils import prompt_sort3,extract_sorted_candidates

def request_gpt(input, output):

    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
    except Exception as e:
        print(f"{Fore.RED}无法打开输入文件: {e}{Style.RESET_ALL}")
        return

    processed_count = 0

    for i, item in enumerate(json_data):
        print(f"\n<--------------------------------------------示例 {i + 1} (line_withRefer_T5)：-------------------------------------------->")

        try:
            # 三个候选
            line_refs = item.get("line_withRefer_T5", [])
            if len(line_refs) < 3:
                print(f"{Fore.YELLOW}Warning: line_withRefer_T5 不足 3 个候选。继续处理（会用空占位）。{Style.RESET_ALL}")
                line_refs = line_refs + [''] * (3 - len(line_refs))

            candidate1, candidate2, candidate3= line_refs[:3]

            # 构造 diff_conflict（你指定的版本）
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + \
                       '\nbase_content\n' + item.get('o_contents', '') + \
                       '\nincoming_change\n' + item.get('b_contents', '')

            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<') \
                                    .replace('base_content', '|||||||') \
                                    .replace('incoming_change', '=======') + '\n>>>>>>>'

            # 模型真实 resolution（你传给 Qwen 做提示用）
            res_region = item.get("res_region", "")

            # 构造 prompt
            prompt = prompt_sort3(candidate1, candidate2, candidate3, diff_conflict)

            # 调模型
            model_output_list = qwen(question=prompt, number=1, truth=res_region)

            sorted_candidates = None

            if not model_output_list:
                sorted_candidates = []
            else:
                first = model_output_list[0]

                if isinstance(first, str):
                    raw_text = first.strip()
                    # 若以 '[' 开头，优先 JSON 解析
                    if raw_text.startswith('['):
                        try:
                            parsed = json.loads(raw_text)
                            if isinstance(parsed, list):
                                extracted = []
                                for elem in parsed:
                                    if isinstance(elem, dict) and len(elem) == 1:
                                        extracted.append(list(elem.values())[0])
                                    else:
                                        extracted.append(str(elem))
                                sorted_candidates = extracted
                        except:
                            sorted_candidates = extract_sorted_candidates(raw_text)
                    else:
                        sorted_candidates = extract_sorted_candidates(raw_text)

                elif isinstance(first, list):
                    # 极少情况：第一层就是候选数组
                    extracted = []
                    for elem in first:
                        if isinstance(elem, dict):
                            vals = list(elem.values())
                            if vals:
                                extracted.append(vals[0])
                        else:
                            extracted.append(str(elem))
                    sorted_candidates = extracted

                else:
                    raw_text = str(first)
                    sorted_candidates = extract_sorted_candidates(raw_text)

            if sorted_candidates is None:
                sorted_candidates = []

            # 写回
            item["sorted_T5"] = sorted_candidates

            processed_count += 1

            # 每 10 条保存
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as out:
                    json.dump(json_data, out, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

        except Exception as e:
            print(f"{Fore.RED}处理示例 {i + 1} 时发生异常: {e}{Style.RESET_ALL}")
            item.setdefault("_error", str(e))
            item["sorted_T5"] = []
            continue

    # 最终保存
    with open(output, 'w', encoding='utf-8') as out:
        json.dump(json_data, out, indent=2, ensure_ascii=False)

    print(f"{Fore.GREEN}完成：共处理 {processed_count} 条样本，结果已保存至 {output}{Style.RESET_ALL}")


#############################################
# 5. 执行入口
#############################################

if __name__ == "__main__":
    request_gpt(
        input="YOUR_PROJECT_DIR/produce2/congra/json/aosp.json",
        output="YOUR_PROJECT_DIR/produce2/congra/json/aosp_qwen_sort.json"
    )


### deepseek

In [ ]:
# 调用 DeepSeek 对top5 进行排序
import json
import re
from colorama import Fore, Style
from deepseek_api import deepSeek
from utils import prompt_sort5,extract_sorted_candidates

def request_gpt(input, output):

    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
    except Exception as e:
        print(f"{Fore.RED}无法打开输入文件: {e}{Style.RESET_ALL}")
        return

    processed_count = 0

    for i, item in enumerate(json_data):
        print(f"\n<--------------------------------------------示例 {i + 1} (line_withRefer_T5)：-------------------------------------------->")

        try:
            # 三个候选
            line_refs = item.get("line_withRefer_T5", [])
            if len(line_refs) < 5:
                print(f"{Fore.YELLOW}Warning: line_withRefer_T5 不足 5 个候选。继续处理（会用空占位）。{Style.RESET_ALL}")
                line_refs = line_refs + [''] * (5 - len(line_refs))

            candidate1, candidate2, candidate3, candidate4, candidate5 = line_refs[:5]

            # 构造 diff_conflict（你指定的版本）
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + \
                       '\nbase_content\n' + item.get('o_contents', '') + \
                       '\nincoming_change\n' + item.get('b_contents', '')

            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<') \
                                    .replace('base_content', '|||||||') \
                                    .replace('incoming_change', '=======') + '\n>>>>>>>'

            # 模型真实 resolution（你传给 Qwen 做提示用）
            res_region = item.get("res_region", "")

            # 构造 prompt
            prompt = prompt_sort5(candidate1, candidate2, candidate3, candidate4, candidate5, diff_conflict)

            # 调模型
            model_output_list = deepSeek(question=prompt, number=1, truth=res_region)

            sorted_candidates = None

            if not model_output_list:
                sorted_candidates = []
            else:
                first = model_output_list[0]

                if isinstance(first, str):
                    raw_text = first.strip()
                    # 若以 '[' 开头，优先 JSON 解析
                    if raw_text.startswith('['):
                        try:
                            parsed = json.loads(raw_text)
                            if isinstance(parsed, list):
                                extracted = []
                                for elem in parsed:
                                    if isinstance(elem, dict) and len(elem) == 1:
                                        extracted.append(list(elem.values())[0])
                                    else:
                                        extracted.append(str(elem))
                                sorted_candidates = extracted
                        except:
                            sorted_candidates = extract_sorted_candidates(raw_text)
                    else:
                        sorted_candidates = extract_sorted_candidates(raw_text)

                elif isinstance(first, list):
                    # 极少情况：第一层就是候选数组
                    extracted = []
                    for elem in first:
                        if isinstance(elem, dict):
                            vals = list(elem.values())
                            if vals:
                                extracted.append(vals[0])
                        else:
                            extracted.append(str(elem))
                    sorted_candidates = extracted

                else:
                    raw_text = str(first)
                    sorted_candidates = extract_sorted_candidates(raw_text)

            if sorted_candidates is None:
                sorted_candidates = []

            # 写回
            item["sorted_T5"] = sorted_candidates

            processed_count += 1

            # 每 10 条保存
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as out:
                    json.dump(json_data, out, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

        except Exception as e:
            print(f"{Fore.RED}处理示例 {i + 1} 时发生异常: {e}{Style.RESET_ALL}")
            item.setdefault("_error", str(e))
            item["sorted_T5"] = []
            continue

    # 最终保存
    with open(output, 'w', encoding='utf-8') as out:
        json.dump(json_data, out, indent=2, ensure_ascii=False)

    print(f"{Fore.GREEN}完成：共处理 {processed_count} 条样本，结果已保存至 {output}{Style.RESET_ALL}")


#############################################
# 5. 执行入口
#############################################

if __name__ == "__main__":
    request_gpt(
        input="YOUR_PROJECT_DIR/deepseek_produce/50repo/json/fred.json",
        output="YOUR_PROJECT_DIR/deepseek_produce/50repo/json/fred_ds_sort.json"
    )


# 消融实验

## 消融实验1: 移除语义相似度（只用base惩罚、token奖励、长度惩罚）

In [ ]:
# 消融实验1: 移除语义相似度（只用base惩罚、token奖励、长度惩罚）
import json
from difflib import SequenceMatcher

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    """判断 target 是否含有 source 中独有 token"""
    return any(tok.strip() and tok in target for tok in source.split())

def score_candidate_ablation1(candidate, a, b, o):
    score = 0

    # 移除基础语义相似度

    # base版本越像，说明没融合，做惩罚
    base_similarity = sim(candidate, o)
    score -= 0.7 * base_similarity

    # 额外结构特征规则
    if contains_tokens(a, candidate) and contains_tokens(b, candidate):
        score += 2  # 代表跨分支融合
    elif contains_tokens(a, candidate) or contains_tokens(b, candidate):
        score += 0.5  # 至少来自其中一个版本

    # 过短代码惩罚（避免损坏代码）
    score -= abs(len(a) - len(candidate)) * 0.0005

    return score

input_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred.json"
output_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred_sorted_1.json"

with open(input_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

for item in dataset:
    if "line_withRefer_T5" not in item:
        continue

    a = item.get("a_contents", "")
    b = item.get("b_contents", "")
    o = item.get("o_contents", "")

    candidates = item["line_withRefer_T5"]
    
    ranked = sorted(
        candidates, 
        key=lambda c: score_candidate_ablation1(c, a, b, o), 
        reverse=True
    )

    item["line_withRefer_T5"] = ranked

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print("消融1排序完成 →", output_file)

## 消融实验2: 移除base惩罚（只用语义相似度、token奖励、长度惩罚）
import json

In [ ]:
# 消融实验2: 移除base惩罚（只用语义相似度、token奖励、长度惩罚）
import json
from difflib import SequenceMatcher

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    """判断 target 是否含有 source 中独有 token"""
    return any(tok.strip() and tok in target for tok in source.split())

def score_candidate_ablation2(candidate, a, b, o):
    score = 0

    # 基础语义相似度
    score += 0.5 * sim(candidate, a)
    score += 0.5 * sim(candidate, b)

    # 移除base版本越像，说明没融合，做惩罚

    # 额外结构特征规则
    if contains_tokens(a, candidate) and contains_tokens(b, candidate):
        score += 2  # 代表跨分支融合
    elif contains_tokens(a, candidate) or contains_tokens(b, candidate):
        score += 0.5  # 至少来自其中一个版本

    # 过短代码惩罚（避免损坏代码）
    score -= abs(len(a) - len(candidate)) * 0.0005

    return score

input_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred.json"
output_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred_sorted_2.json"

with open(input_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

for item in dataset:
    if "line_withRefer_T5" not in item:
        continue

    a = item.get("a_contents", "")
    b = item.get("b_contents", "")
    o = item.get("o_contents", "")

    candidates = item["line_withRefer_T5"]
    
    ranked = sorted(
        candidates, 
        key=lambda c: score_candidate_ablation2(c, a, b, o), 
        reverse=True
    )

    item["line_withRefer_T5"] = ranked

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print("消融2排序完成 →", output_file)

## 消融实验3: 移除token奖励（只用语义相似度、base惩罚、长度惩罚）

In [ ]:
# 消融实验3: 移除token奖励（只用语义相似度、base惩罚、长度惩罚）
import json
from difflib import SequenceMatcher

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    """判断 target 是否含有 source 中独有 token"""
    return any(tok.strip() and tok in target for tok in source.split())

def score_candidate_ablation3(candidate, a, b, o):
    score = 0

    # 基础语义相似度
    score += 0.5 * sim(candidate, a)
    score += 0.5 * sim(candidate, b)

    # base版本越像，说明没融合，做惩罚
    base_similarity = sim(candidate, o)
    score -= 0.7 * base_similarity

    # 移除额外结构特征规则

    # 过短代码惩罚（避免损坏代码）
    score -= abs(len(a) - len(candidate)) * 0.0005

    return score

input_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred.json"
output_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred_sorted_3.json"

with open(input_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

for item in dataset:
    if "line_withRefer_T5" not in item:
        continue

    a = item.get("a_contents", "")
    b = item.get("b_contents", "")
    o = item.get("o_contents", "")

    candidates = item["line_withRefer_T5"]
    
    ranked = sorted(
        candidates, 
        key=lambda c: score_candidate_ablation3(c, a, b, o), 
        reverse=True
    )

    item["line_withRefer_T5"] = ranked

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print("消融3排序完成 →", output_file)

## 消融实验4: 移除长度惩罚（只用语义相似度、base惩罚、token奖励）

In [ ]:
# 消融实验4: 移除长度惩罚（只用语义相似度、base惩罚、token奖励）
import json
from difflib import SequenceMatcher

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    """判断 target 是否含有 source 中独有 token"""
    return any(tok.strip() and tok in target for tok in source.split())

def score_candidate_ablation4(candidate, a, b, o):
    score = 0

    # 基础语义相似度
    score += 0.5 * sim(candidate, a)
    score += 0.5 * sim(candidate, b)

    # base版本越像，说明没融合，做惩罚
    base_similarity = sim(candidate, o)
    score -= 0.7 * base_similarity

    # 额外结构特征规则
    if contains_tokens(a, candidate) and contains_tokens(b, candidate):
        score += 2  # 代表跨分支融合
    elif contains_tokens(a, candidate) or contains_tokens(b, candidate):
        score += 0.5  # 至少来自其中一个版本

    # 移除过短代码惩罚（避免损坏代码）

    return score

input_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred.json"
output_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred_sorted_4.json"

with open(input_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

for item in dataset:
    if "line_withRefer_T5" not in item:
        continue

    a = item.get("a_contents", "")
    b = item.get("b_contents", "")
    o = item.get("o_contents", "")

    candidates = item["line_withRefer_T5"]
    
    ranked = sorted(
        candidates, 
        key=lambda c: score_candidate_ablation4(c, a, b, o), 
        reverse=True
    )

    item["line_withRefer_T5"] = ranked

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print("消融4排序完成 →", output_file)

## 消融实验5: 移除token奖励和长度惩罚（只用语义相似度、base惩罚）

In [ ]:
# 消融实验5: 移除token奖励和长度惩罚（只用语义相似度、base惩罚）
import json
from difflib import SequenceMatcher

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    """判断 target 是否含有 source 中独有 token"""
    return any(tok.strip() and tok in target for tok in source.split())

def score_candidate_ablation5(candidate, a, b, o):
    score = 0

    # 基础语义相似度
    score += 0.5 * sim(candidate, a)
    score += 0.5 * sim(candidate, b)

    # base版本越像，说明没融合，做惩罚
    base_similarity = sim(candidate, o)
    score -= 0.7 * base_similarity

    # 移除额外结构特征规则
    # 移除过短代码惩罚（避免损坏代码）

    return score

input_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred.json"
output_file = r"YOUR_PROJECT_DIR\produce2\50repo\json\fred_sorted_5.json"

with open(input_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

for item in dataset:
    if "line_withRefer_T5" not in item:
        continue

    a = item.get("a_contents", "")
    b = item.get("b_contents", "")
    o = item.get("o_contents", "")

    candidates = item["line_withRefer_T5"]
    
    ranked = sorted(
        candidates, 
        key=lambda c: score_candidate_ablation5(c, a, b, o), 
        reverse=True
    )

    item["line_withRefer_T5"] = ranked

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print("消融5排序完成 →", output_file)

# 迭代（废弃）

In [ ]:
import json
import random
from utils import PromptRS3
from colorama import Fore, Style
from alibaba_api import qwen
from database_api_T5_BM25_L2 import query_similar

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"

# 加载 generation1.json 文件
def load_generation_samples(generation_file):
    """加载 generation1.json 中的样本数据"""
    with open(generation_file, 'r', encoding='utf-8') as f:
        return json.load(f)

def get_top3_samples(generation_data, conflict_id):
    """根据 conflict_id 获取前3个代表性样本"""
    conflict_key = f"conflict_{conflict_id}"
    
    if conflict_key not in generation_data:
        print(f"{Fore.YELLOW}Warning: conflict_{conflict_id} not found in generation1.json{Style.RESET_ALL}")
        return "", "", ""
    
    samples = generation_data[conflict_key]
    
    # 获取前三个样本的代码
    attempt1 = samples[0]['code'] if len(samples) > 0 else ""
    attempt2 = samples[1]['code'] if len(samples) > 1 else ""
    attempt3 = samples[2]['code'] if len(samples) > 2 else ""
    
    return attempt1, attempt2, attempt3

def request_gpt(input, output, generation_file):
    # 加载 generation1.json
    generation_data = load_generation_samples(generation_file)
    
    try:
        with open(input, 'r', encoding='utf-8') as input_file:
            json_data = json.load(input_file)
        
        #random.shuffle(json_data) # 随机洗牌顺序
        processed_count = 0  # 用于记录已处理的数据数量
        
        for i, item in enumerate(json_data):
            if i + 1 < 208: # 从异常发生点接着运行
               continue
            if i + 1 ==502: # 从异常发生点接着运行
                break

            id = item.get('id', '')
            repo_url = item.get('repo', '')
            repo_name = repo_url.rstrip('/').split('/')[-1].split('-')[0]
            res_region = item.get('res_region', '')
            
            conflict = 'concurrent_change\n' + item.get('a_contents', '') + '\nbase_content\n' + item.get('o_contents', '') + '\nincoming_change\n' + item.get('b_contents', '')

            query = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            global_id, related_c, related_r = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)
            diff_conflict = conflict.replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'

            # 查找相似冲突
            refer_c1 = ""
            refer_r1 = ""
            for j in range(len(global_id)):  # 找到第一个 != id 的索引
                if global_id[j] != id:
                    print(f"<<<<<<我的id是{id}, 相似冲突id是{global_id[j]}>>>>>>")
                    refer_c1 = related_c[j].replace(' _newline_ ', '\n').replace('concurrent_change', '<<<<<<<').replace('base_content', '|||||||').replace('incoming_change', '=======') + '\n>>>>>>>'
                    refer_r1 = related_r[j].replace('_newline_', '\n')
                    break

            # 从 generation1.json 获取前三个样本
            existing_attempt1, existing_attempt2, existing_attempt3 = get_top3_samples(generation_data, id)
            
            print(f"\n<--------------------------------------------示例{i + 1}(line_withRS1)：-------------------------------------------->")
            print(f"Conflict ID: {id}")
            # if existing_attempt1:
            #     print(f"Existing Attempt 1 (rank 1): {existing_attempt1[:100]}...")
            # if existing_attempt2:
            #     print(f"Existing Attempt 2 (rank 2): {existing_attempt2[:100]}...")
            # if existing_attempt3:
            #     print(f"Existing Attempt 3 (rank 3): {existing_attempt3[:100]}...")
            
            item['line_withRS1'] = qwen(
                question=PromptRS3(diff_conflict, existing_attempt1, existing_attempt2, existing_attempt3, refer_c1, refer_r1), 
                number=3, 
                truth=res_region
            )
            
            # 每处理完一条数据，计数器加1
            processed_count += 1
            
            # 每处理10条数据，自动保存一次
            if processed_count % 10 == 0:
                with open(output, 'w', encoding='utf-8') as output_file:
                    json.dump(json_data, output_file, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}✓ 自动保存: 已处理 {processed_count} 条数据 (当前示例 {i + 1}){Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error----:An error occurred: {e}{Style.RESET_ALL}")
    finally:
        with open(output, 'w', encoding='utf-8') as output_file:
            json.dump(json_data, output_file, indent=2, ensure_ascii=False)
        print(f"{Fore.GREEN}Warnning----:ChatGPT已运行数据已保存! 异常发生点:第{i + 1}示例{Style.RESET_ALL}")
        output_file.close()
    
request_gpt(
    input="YOUR_PROJECT_DIR/produce/50repo/json/fredRS3.json",
    output="YOUR_PROJECT_DIR/produce/50repo/json/fredRS3.json",
    generation_file="YOUR_PROJECT_DIR/produce/fred_generation00.json"
) 

## Unified generation pipeline

Edit only the `config` dictionary in the next cell:

- `dataset_group`: dataset folder under `dataset/`
- `repo_name`: JSON file name without `.json`
- `model_name`: `qwen`, `deepseek`, `gpt35`, or `gpt4o`
- `num_candidates`: how many candidates one model call should generate
- `top_k`: how many reranked candidates to keep
- `sample_limit`: how many records to process in this run
- `start_index`: where to start in the dataset
- `use_similar_history`: whether to retrieve the most similar historical conflict from the vector database
- `vector_database_mode`: `auto`, `fixed`, `repo_field`, or `repo_field_time`
- `vector_database_name`: required only when `vector_database_mode` is `fixed`
- `cluster_endpoint` and `token`: required when `use_similar_history` is enabled

In [ ]:
from run_end_to_end_pipeline import run_pipeline, print_summary

# ===== 可按需调整 =====
payload = run_pipeline(
    input_file="dataset/mergebert/json/autopsy.json",
    model_name="qwen",
    sample_limit=700,
    start_index=0,
    num_candidates=5,
    top_k=3,
    truth_field="res_region",
    use_rerank=True,
    use_similar_history=False,
    output_base="dataset/mergebert/json/autopsy_qwen_e2e",
    record_file="metric/evaluation_records.jsonl",
)

print_summary(payload)

In [ ]:
# 端到端一键实验：多候选 -> rerank(可关) -> 评估
from run_end_to_end_pipeline import run_pipeline, print_summary

# ===== 可按需调整（默认前200条） =====
payload = run_pipeline(
    input_file="dataset/mergebert/json/autopsy.json",
    model_name="qwen",
    sample_limit=200,
    start_index=0,
    num_candidates=5,
    top_k=3,
    truth_field="res_region",
    use_rerank=False,  # 关闭重排序做对照实验
    use_similar_history=False,
    output_base="dataset/mergebert/json/autopsy_no_mask_200_qwen_e2e",
    record_file="metric/evaluation_records.jsonl",
)

print_summary(payload)

# dataset_rerank pipeline

> 请先确认以下内容：
> - API token/endpoint 已在代码中更新为可用值
> - 需要写入向量数据库的 collection 是否允许覆盖

## 1. Split merged_sorted.json into train/test (top/bottom 80/60/40/20)

In [ ]:
import json
from pathlib import Path

source_path = Path("dataset_rerank/merged_sorted.json")
train_dir = Path("dataset_rerank/train")
test_dir = Path("dataset_rerank/test")

train_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

with open(source_path, "r", encoding="utf-8") as f:
    data = json.load(f)

n = len(data)
if n == 0:
    raise ValueError("merged_sorted.json is empty")

ratios = [0.8, 0.6, 0.4, 0.2]
for ratio in ratios:
    top_count = max(1, int(n * ratio))
    top = data[:top_count]
    bottom = data[top_count:]

    top_name = train_dir / f"merged_top{int(ratio * 100)}.json"
    bottom_name = test_dir / f"merged_bottom{int((1 - ratio) * 100)}.json"

    with open(top_name, "w", encoding="utf-8") as f:
        json.dump(top, f, indent=2, ensure_ascii=False)
    with open(bottom_name, "w", encoding="utf-8") as f:
        json.dump(bottom, f, indent=2, ensure_ascii=False)

    print(f"saved {top_name} ({len(top)}) and {bottom_name} ({len(bottom)})")

## 2. Convert time JSON files to vector files (dataset_rerank/vector)

In [ ]:
import json
from pathlib import Path

def json_to_vector(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as file:
        data_list = json.load(file)

    valid_records_count = 0
    with open(output_path, "w", encoding="utf-8") as output_file:
        for data in data_list:
            res_label = data.get("res_label", "")
            if res_label == "null":
                continue
            a_contents = data.get("a_contents", "")
            o_contents = data.get("o_contents", "")
            b_contents = data.get("b_contents", "")
            res_region = data.get("res_region", "")
            global_id = data.get("id", "")
            timestamp = data.get("timestamp", "")

            conflict = (
                "concurrent_change\n" + a_contents +
                "\nbase_content\n" + o_contents +
                "\nincoming_change\n" + b_contents
            )
            processed_conflict = conflict.replace("\n", " _newline_ ").replace("\t", " ")
            processed_res = res_region.replace("\n", " _newline_ ").replace("\t", " ")
            output_file.write(
                processed_conflict + "\t" + processed_res + "\t" + str(global_id) + "\t" + timestamp + "\n"
            )
            valid_records_count += 1

    print(f"processed {input_path} -> {output_path} ({valid_records_count} rows)")

vector_dir = Path("dataset_rerank/vector")
vector_dir.mkdir(parents=True, exist_ok=True)

json_candidates = set()
json_candidates.update(Path("dataset_rerank/train").glob("*time*.json"))
json_candidates.update(Path("dataset_rerank/test").glob("*time*.json"))
json_candidates.update(Path("dataset_rerank").glob("*time*.json"))

if not json_candidates:
    print("no *time*.json files found under dataset_rerank")
else:
    for json_path in sorted(json_candidates):
        output_path = vector_dir / f"{json_path.stem}_Conflicts.txt"
        json_to_vector(json_path, output_path)

## 3. Insert dataset_rerank/vector files into Milvus (T5-BM25-L2)

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import torch
from pymilvus import MilvusClient, DataType, Function, FunctionType
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT"
TOKEN = "YOUR_MILVUS_TOKEN"

DROP_EXISTING = False
GROUP_ID = 99  # dataset_rerank group id

client = MilvusClient(uri=CLUSTER_ENDPOINT, token=TOKEN)

def ensure_collection(collection_name):
    if collection_name in client.list_collections():
        if DROP_EXISTING:
            client.drop_collection(collection_name=collection_name)
        else:
            return
    schema = MilvusClient.create_schema(auto_id=True, enable_dynamic_field=True)
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="global_id", datatype=DataType.INT32)
    schema.add_field(field_name="chunk_content", datatype=DataType.VARCHAR, max_length=2048, enable_analyzer=True)
    schema.add_field(field_name="chunk_resolution", datatype=DataType.VARCHAR, max_length=1024)
    schema.add_field(field_name="chunk_timestamp", datatype=DataType.VARCHAR, max_length=64)
    schema.add_field(field_name="chunk_sparse_embedding", datatype=DataType.SPARSE_FLOAT_VECTOR)
    schema.add_field(field_name="chunk_embedding", datatype=DataType.FLOAT_VECTOR, dim=768)

    bm25_function = Function(
        name="text_bm25_emb",
        input_field_names="chunk_content",
        output_field_names="chunk_sparse_embedding",
        function_type=FunctionType.BM25,
    )
    schema.add_function(bm25_function)

    index_params = MilvusClient.prepare_index_params()
    index_params.add_index(
        field_name="chunk_embedding",
        metric_type="L2",
        index_type="AUTOINDEX",
        index_name="vector_index",
    )
    index_params.add_index(
        field_name="chunk_sparse_embedding",
        index_name="sparse_inverted_index",
        index_type="SPARSE_INVERTED_INDEX",
        metric_type="BM25",
        params={"drop_ratio_build": 0.2},
    )

    client.create_collection(collection_name=collection_name, schema=schema, index_params=index_params)

def load_model(local_model_path="./codeT5/codet5-base"):
    try:
        tokenizer = AutoTokenizer.from_pretrained(local_model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(local_model_path).to("cuda" if torch.cuda.is_available() else "cpu")
        print("using local CodeT5 model")
    except Exception as e:
        print(f"local model load failed: {e}")
        tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")
        model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-base").to("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer.save_pretrained(local_model_path)
        model.save_pretrained(local_model_path)
        print("downloaded CodeT5 model")
    return tokenizer, model

tokenizer, model = load_model()

def insert_vector_file(vector_path):
    repo_name = vector_path.stem.replace("_Conflicts", "")
    ensure_collection(repo_name)

    pair = pd.read_csv(vector_path, sep="\t", header=None).drop_duplicates()
    conflicts = pair[0].fillna("").values
    res_labels = pair[1].fillna("").values
    global_ids = pair[2].fillna(0).astype(int).values
    timestamps = pair[3].fillna("").values if pair.shape[1] > 3 else [""] * len(pair)

    data_pairs = list(zip(conflicts, res_labels, global_ids, timestamps))
    for conflict, resolution, g_id, timestamp in tqdm(data_pairs, desc=f"{repo_name}"):
        inputs = tokenizer(conflict, padding=True, truncation=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            enc = model.encoder(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            embeddings = enc.last_hidden_state.mean(dim=1).cpu().numpy()
        encode = embeddings[0]

        res_data = resolution if isinstance(resolution, str) and resolution.strip() else "_newline_"
        client.insert(
            collection_name=repo_name,
            data=[
                {
                    "group_id": GROUP_ID,
                    "global_id": int(g_id),
                    "chunk_content": str(conflict).encode("utf-8")[:2048].decode("utf-8", errors="ignore"),
                    "chunk_resolution": str(res_data).encode("utf-8")[:1024].decode("utf-8", errors="ignore"),
                    "chunk_timestamp": str(timestamp),
                    "chunk_embedding": encode,
                }
            ],
        )
    print(f"inserted {len(data_pairs)} rows into {repo_name}")

vector_dir = Path("dataset_rerank/vector")
vector_files = sorted(vector_dir.glob("*_Conflicts.txt"))
if not vector_files:
    print("no vector files found under dataset_rerank/vector")
else:
    for vector_file in vector_files:
        insert_vector_file(vector_file)

## 4. Generate line_no_T5 / line_withRefer_T5 / line_withrerankT5 (deepseek/qwen/gpt)

In [ ]:
import json
from difflib import SequenceMatcher
from pathlib import Path
from colorama import Fore, Style
from utils import simplePrompt, prompt1
from alibaba_api import qwen
from deepseek_api import deepSeek
from openai_api import gpt_35
from database_api_T5_BM25_L2 import query_similarT5

CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT"  # Serverless-T5
TOKEN = "YOUR_MILVUS_TOKEN"

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    return any(tok.strip() and tok in target for tok in source.split())

def normalize_refer_conflict(raw_conflict):
    return (
        raw_conflict.replace(" _newline_ ", "\n")
        .replace("concurrent_change", "<<<<<<<")
        .replace("base_content", "|||||||")
        .replace("incoming_change", "=======")
        + "\n>>>>>>>"
    )

def strip_markers(text):
    return (
        text.replace("<<<<<<<", "")
        .replace("|||||||", "")
        .replace("=======", "")
        .replace(">>>>>>>", "")
    )

def score_reference(refer_conflict, a, b, o):
    score = 0
    base_text = strip_markers(refer_conflict)
    score += 0.5 * sim(base_text, a)
    score += 0.5 * sim(base_text, b)
    score -= 0.7 * sim(base_text, o)
    if contains_tokens(a, base_text) and contains_tokens(b, base_text):
        score += 1.0
    elif contains_tokens(a, base_text) or contains_tokens(b, base_text):
        score += 0.3
    score -= abs(len(base_text) - (len(a) + len(b))) * 0.0001
    return score

def pick_best_reference(related_cs, related_rs, a, b, o, current_id, global_ids):
    candidates = []
    for idx, raw_c in enumerate(related_cs):
        if global_ids[idx] == current_id:
            continue
        refer_c = normalize_refer_conflict(raw_c)
        refer_r = related_rs[idx].replace("_newline_", "\n")
        candidates.append((refer_c, refer_r))
    if not candidates:
        return "", ""
    ranked = sorted(
        candidates,
        key=lambda pair: score_reference(pair[0], a, b, o),
        reverse=True,
    )
    return ranked[0]

def request_model(input_path, output_path, model_func, num_candidates=3, top_k_similar=5, sample_limit=None, start_index=0):
    try:
        with open(input_path, "r", encoding="utf-8") as input_file:
            json_data = json.load(input_file)

        processed = 0
        for i, item in enumerate(json_data):
            if i < start_index:
                continue
            if sample_limit is not None and processed >= sample_limit:
                break

            current_id = item.get("id", "")
            repo_url = item.get("repo", "")
            repo_name = repo_url.rstrip("/").split("/")[-1].split("-")[0] if repo_url else ""
            res_region = item.get("res_region", "")

            conflict = (
                "concurrent_change\n" + item.get("a_contents", "") +
                "\nbase_content\n" + item.get("o_contents", "") +
                "\nincoming_change\n" + item.get("b_contents", "")
            )
            diff_conflict = conflict.replace("concurrent_change", "<<<<<<<").replace("base_content", "|||||||").replace("incoming_change", "=======") + "\n>>>>>>>"
            query = conflict.replace("\n", " _newline_ ").replace("\t", " ")

            global_ids, related_cs, related_rs = query_similarT5(CLUSTER_ENDPOINT, TOKEN, repo_name, query, top_k_similar)

            # line_no_T5 (nothing)
            item["line_no_T5"] = model_func(question=simplePrompt(diff_conflict), number=num_candidates, truth=res_region)

            # line_withRefer_T5 (top1 similar)
            refer_c1, refer_r1 = "", ""
            for j in range(len(global_ids)):
                if global_ids[j] != current_id:
                    refer_c1 = normalize_refer_conflict(related_cs[j])
                    refer_r1 = related_rs[j].replace("_newline_", "\n")
                    break
            item["line_withRefer_T5"] = model_func(question=prompt1(refer_c1, refer_r1, diff_conflict), number=num_candidates, truth=res_region)

            # line_withrerankT5 (rerank top5 similar)
            best_c, best_r = pick_best_reference(related_cs, related_rs, item.get("a_contents", ""), item.get("b_contents", ""), item.get("o_contents", ""), current_id, global_ids)
            item["line_withrerankT5"] = model_func(question=prompt1(best_c, best_r, diff_conflict), number=num_candidates, truth=res_region)

            processed += 1
            if processed % 10 == 0:
                with open(output_path, "w", encoding="utf-8") as out:
                    json.dump(json_data, out, indent=2, ensure_ascii=False)
                print(f"{Fore.GREEN}auto-save {processed} items{Style.RESET_ALL}")

    except Exception as e:
        print(f"{Fore.RED}Error: {e}{Style.RESET_ALL}")
    finally:
        with open(output_path, "w", encoding="utf-8") as out:
            json.dump(json_data, out, indent=2, ensure_ascii=False)
        print(f"{Fore.GREEN}saved to {output_path}{Style.RESET_ALL}")

# ===== example configs =====
jobs = [
    {
        "name": "qwen",
        "model": qwen,
        "input": "dataset_rerank/train/merged_top80.json",
        "output": "dataset_rerank/outputs/qwen/merged_top80_qwen.json",
        "num_candidates": 5,
    },
    {
        "name": "deepseek",
        "model": deepSeek,
        "input": "dataset_rerank/train/merged_top80.json",
        "output": "dataset_rerank/outputs/deepseek/merged_top80_deepseek.json",
        "num_candidates": 5,
    },
    {
        "name": "gpt",
        "model": gpt_35,
        "input": "dataset_rerank/train/merged_top80.json",
        "output": "dataset_rerank/outputs/gpt/merged_top80_gpt.json",
        "num_candidates": 3,
    },
 ]

for job in jobs:
    out_path = Path(job["output"])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    request_model(
        input_path=job["input"],
        output_path=job["output"],
        model_func=job["model"],
        num_candidates=job["num_candidates"],
        top_k_similar=5,
        sample_limit=500,
        start_index=0,
    )

## 5. Post-rerank generated candidates with semantic reranker

In [ ]:
import json
from difflib import SequenceMatcher
from pathlib import Path

def sim(a, b):
    if not a or not b:
        return 0
    return SequenceMatcher(None, a, b).ratio()

def contains_tokens(source, target):
    return any(tok.strip() and tok in target for tok in source.split())

def score_candidate(candidate, a, b, o):
    score = 0
    score += 0.5 * sim(candidate, a)
    score += 0.5 * sim(candidate, b)
    score -= 0.7 * sim(candidate, o)
    if contains_tokens(a, candidate) and contains_tokens(b, candidate):
        score += 2
    elif contains_tokens(a, candidate) or contains_tokens(b, candidate):
        score += 0.5
    score -= abs(len(a) - len(candidate)) * 0.0005
    return score

def rerank_json_file(input_file, output_file, field_name):
    with open(input_file, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    for item in dataset:
        if field_name not in item:
            continue
        a = item.get("a_contents", "")
        b = item.get("b_contents", "")
        o = item.get("o_contents", "")
        candidates = item[field_name]
        if not isinstance(candidates, list):
            continue
        item[field_name] = sorted(
            candidates,
            key=lambda c: score_candidate(c, a, b, o),
            reverse=True,
        )

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

    print(f"reranked {field_name} -> {output_file}")

output_root = Path("dataset_rerank/outputs")
fields_to_rerank = ["line_withRefer_T5", "line_withrerankT5"]

for input_path in output_root.rglob("*.json"):
    for field_name in fields_to_rerank:
        out_path = input_path.with_name(f"{input_path.stem}_{field_name}_reranked.json")
        rerank_json_file(input_path, out_path, field_name)

## 6. Exact-match evaluation + summary.md

In [ ]:
import json
from pathlib import Path

def exact_match_hit(candidates, truth):
    if truth is None:
        return False
    truth_str = str(truth).strip()
    if not truth_str:
        return False
    if isinstance(candidates, list):
        return any(str(c).strip() == truth_str for c in candidates)
    return str(candidates).strip() == truth_str

def evaluate_file(path, fields):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = {field: {"hit": 0, "total": 0} for field in fields}
    for item in data:
        truth = item.get("res_region", "")
        for field in fields:
            if field not in item:
                continue
            results[field]["total"] += 1
            if exact_match_hit(item[field], truth):
                results[field]["hit"] += 1
    return results

output_root = Path("dataset_rerank/outputs")
fields = ["line_no_T5", "line_withRefer_T5", "line_withrerankT5"]

rows = []
overall = {field: {"hit": 0, "total": 0} for field in fields}

for json_path in sorted(output_root.rglob("*.json")):
    file_results = evaluate_file(json_path, fields)
    for field in fields:
        overall[field]["hit"] += file_results[field]["hit"]
        overall[field]["total"] += file_results[field]["total"]
    rows.append((json_path, file_results))

summary_path = Path("dataset_rerank/summary.md")
summary_path.parent.mkdir(parents=True, exist_ok=True)

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("# dataset_rerank exact-match summary\n\n")
    f.write("## Per file\n\n")
    f.write("| File | Field | Hit | Total | Accuracy |\n")
    f.write("| --- | --- | --- | --- | --- |\n")
    for file_path, file_results in rows:
        rel_path = file_path.as_posix()
        for field in fields:
            hit = file_results[field]["hit"]
            total = file_results[field]["total"]
            acc = (hit / total) if total else 0
            f.write(f"| {rel_path} | {field} | {hit} | {total} | {acc:.4f} |\n")
    f.write("\n## Overall\n\n")
    f.write("| Field | Hit | Total | Accuracy |\n")
    f.write("| --- | --- | --- | --- |\n")
    for field in fields:
        hit = overall[field]["hit"]
        total = overall[field]["total"]
        acc = (hit / total) if total else 0
        f.write(f"| {field} | {hit} | {total} | {acc:.4f} |\n")

print(f"summary saved -> {summary_path}")